# Hypothesis 3 Justice Principles: Manipulation Outcomes Analysis

This notebook mirrors the Hypothesis 6 descriptive styling to highlight manipulator outcomes when they are tasked with rescuing the least popular justice principle from the Phase 1 rankings.

**Design System Notes**  
- Bayreuth Green `#009260` anchors positive and consensus outcomes.  
- Dark Gray `#48535A`, Medium Gray `#7F8990`, and Light Gray `#EBEBE4` support typography, baselines, and grid work.  
- Apply horizontal layouts and compact tables to keep cross-cohort comparisons legible.

In [1]:
import json
from collections import OrderedDict
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import sys

_NOTEBOOK_DIR = Path.cwd().resolve()
_REPO_ROOT = None
for candidate in [_NOTEBOOK_DIR, *_NOTEBOOK_DIR.parents]:
    if (candidate / "hypothesis_testing").exists() and (candidate / "config").exists():
        _REPO_ROOT = candidate
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

if _REPO_ROOT is None:
    raise RuntimeError("Cannot locate repository root from notebook path.")

from hypothesis_testing.utils_hypothesis_testing.style import (
    apply_bayreuth_theme,
    format_principle_label,
)

apply_bayreuth_theme()

pd.set_option("display.precision", 1)


In [2]:

RESULTS_BASE = _REPO_ROOT / "hypothesis_testing" / "hypothesis_3" / "results"

INTELLIGENCE_LEVELS = OrderedDict([
    ("low", "Low Intelligence Manipulator"),
    ("high", "High Intelligence Manipulator"),
])

PRINCIPLE_CANONICAL_TO_DISPLAY = OrderedDict([
    ("maximizing_average_floor_constraint", "Max Avg + Floor"),
    ("maximizing_average", "Max Avg Income"),
    ("maximizing_average_range_constraint", "Max Avg + Range"),
    ("maximizing_floor", "Max Floor"),
])


def load_result_payload(path: Path) -> Dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def summarise_manipulation_outcomes(intelligence_key: str) -> tuple[pd.DataFrame, List[str]]:
    results_dir = RESULTS_BASE / intelligence_key
    if not results_dir.exists():
        raise FileNotFoundError(f"Results directory not found: {results_dir}")

    counts: Dict[str, Dict[str, int]] = {
        canonical: {"attempts": 0, "success": 0}
        for canonical in PRINCIPLE_CANONICAL_TO_DISPLAY
    }
    errors: List[str] = []

    for result_path in sorted(results_dir.glob("*_results.json")):
        try:
            payload = load_result_payload(result_path)
        except json.JSONDecodeError as exc:
            errors.append(f"{result_path.name}: JSON decode error ({exc})")
            continue

        general_info = payload.get("general_information", {})
        target_info = general_info.get("manipulator_target_info")

        if not target_info:
            errors.append(f"{result_path.name}: missing manipulator_target_info")
            continue
        if not target_info.get("delivered", False):
            errors.append(f"{result_path.name}: target delivery failed")
            continue

        target_principle = target_info.get("target_principle")
        if target_principle not in PRINCIPLE_CANONICAL_TO_DISPLAY:
            errors.append(
                f"{result_path.name}: unexpected target_principle '{target_principle}'"
            )
            continue

        counts[target_principle]["attempts"] += 1

        consensus_reached = bool(general_info.get("consensus_reached", False))
        consensus_principle = general_info.get("consensus_principle")
        success = consensus_reached and consensus_principle == target_principle
        if success:
            counts[target_principle]["success"] += 1

    rows: List[Dict[str, float | int | str]] = []
    for canonical, display_name in PRINCIPLE_CANONICAL_TO_DISPLAY.items():
        attempts = counts[canonical]["attempts"]
        successes = counts[canonical]["success"]
        share = (successes / attempts * 100.0) if attempts else np.nan
        rows.append(
            {
                "Least popular Principle": format_principle_label(display_name),
                "Attempted Manipulations": int(attempts),
                "Successful Manipulation": int(successes),
                "Share of Successful Manipulation": share,
            }
        )

    table = pd.DataFrame(rows)
    return table, errors


In [3]:

def format_share(value: float) -> str:
    if pd.isna(value):
        return "—"
    return f"{value:.1f}%"

summary_tables: Dict[str, pd.DataFrame] = {}
error_log: Dict[str, List[str]] = {}

for key in INTELLIGENCE_LEVELS:
    table, errors = summarise_manipulation_outcomes(key)
    summary_tables[key] = table
    error_log[key] = errors

for key, label in INTELLIGENCE_LEVELS.items():
    display(Markdown(f"## {label}"))
    styler = (
        summary_tables[key]
        .style
        .format({"Share of Successful Manipulation": format_share})
        .hide(axis="index")
        .set_properties(
            subset=["Least popular Principle"],
            **{"text-align": "left"},
        )
        .set_properties(
            subset=[
                "Attempted Manipulations",
                "Successful Manipulation",
                "Share of Successful Manipulation",
            ],
            **{"text-align": "right"},
        )
    )
    display(styler)

    if error_log[key]:
        display(
            Markdown(
                f"*⚠️ Skipped {len(error_log[key])} runs due to missing or invalid metadata.*"
            )
        )


## Low Intelligence Manipulator

AttributeError: 'Styler' object has no attribute 'hide_index'